In [1]:
import pandas as pd
import glob
import duckdb
import tqdm
from pathlib import Path
import numpy as np
import os
from trebl_tools import pipelines, finder, umi_deduplicate, initial_map, map_refiner

# 1. Loading in mismatches

In [2]:
activity_comparison = pd.read_csv("../../output/GCN4_sk_vs_ec_activities.csv")
mismatches = activity_comparison[np.abs(activity_comparison["EC_activity"] - activity_comparison["SK_activity"]) > 10]
mismatches[(mismatches["rep"] == 1) & (mismatches["time"] == 0)]

,Unnamed: 0,AD,AD_BC,RPTR_BC,rep,time,EC_AD_umi_count,EC_RT_umi_count,EC_activity,AD_umi_count,RT_umi_count,SK_activity
41730,41730,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,CTGTTGTGAAA,GGAAGAAGTCATGA,1,0,484,125,0.258264,1,123,123.0
41731,41731,CAACAATTGGATAAAGTTCAAGCATTTGTTTCTGGTACTGTTTCTC...,TAAACGTGAAA,TTTCCTAGGCCTTT,1,0,342,76,0.222222,1,74,74.0
41733,41733,TCTACTCCATCATTTGAATCTCCAGGTTACTTCTCTCATGATACTT...,TAAAATCGGGT,GACTCAGAAGAGTC,1,0,124,45,0.362903,1,44,44.0
41734,41734,TATAATAAAGCTTTGCAATGGGGTGATGTTATTAAATTGGTTGGTG...,GTTAAAGCTGT,GTACAAGCGTATCA,1,0,499,54,0.108216,1,55,55.0
41735,41735,GCTAGAAATACTGAAGCTGCTAGAAGATCTAGAGCTAGGAAGATGC...,ATGAATTATTG,CGATATGTATTACA,1,0,868,133,0.153226,1,134,134.0
...,...,...,...,...,...,...,...,...,...,...,...,...
55278,55278,TCTGAATCTAAGAGATCTACTGATTTGGATTCTGCTGTTGAGAACT...,TAACCCTCGCA,AAACTGCAGAAATA,1,0,270,60,0.222222,1,64,64.0
55292,55292,TTGATTTCTGCTTTGCCAGCTATGGAGAAATCTATTAATAGAGCTC...,CCCCCACGCTA,CGCCACACGTCGCA,1,0,6,15,2.500000,1,16,16.0
55295,55295,TTGCCACCAATCTTGGTTGATGAGAATGATCCAGTTGCTTTGAAGA...,ACAGTCGTACT,CTTATTCTAAACCA,1,0,515,33,0.064078,1,33,33.0
55305,55305,GTTACTTTGGAAGATGTTTCTGATTGTGCTAATGCTGTTACTTTGG...,TGAGAGACCAT,CTCTGTTTTAGAAT,1,0,86,16,0.186047,1,16,16.0


In [3]:
mismatches[["rep", "time"]].value_counts()

rep  time
3    240     2339
1    240     2324
3    180     2321
2    240     2320
1    180     2315
2    180     2311
1    30      2172
2    15      2163
     30      2138
3    15      2138
     10      2134
1    10      2126
2    10      2122
1    15      2110
3    30      2098
     5       2050
2    5       2038
1    5       2032
2    0       1972
3    0       1943
1    0       1910
Name: count, dtype: int64

In [4]:
# con = duckdb.connect("../../duckdb/GCN4_final.db", read_only=True)

In [5]:
# tables_df = con.execute("SHOW TABLES").fetchdf()
# tables_df[tables_df["name"].str.contains("trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial")].iloc[0].iloc[0]

In [6]:
# con.execute("""
# SELECT *
# FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
# WHERE AD_BC = 'CTGTTGTGAAA'
# LIMIT 20
# """).fetchdf()

In [7]:
# con.close()

# Testing on one part of AD results

In [8]:
yeast_pool_C_umi_AD_seq_files = glob.glob("/global/scratch/projects/fc_mvslab/data/sequencing/20250218_MZCCSCU_MedGenome/MZ/results/assembled/*1_0*")
# yeast_pool_C_umi_output_path = "/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/TREBL/output/GCN4/yeast_pool_C_umi" 

In [9]:
yeast_pool_C_umi_AD_seq_files

['/global/scratch/projects/fc_mvslab/data/sequencing/20250218_MZCCSCU_MedGenome/MZ/results/assembled/AD_1_0.fastq.gz.assembled.fastq']

In [10]:
EC_AD = finder.Barcode(name = "AD",
                        preceder = "GGCTAGC",
                       post = "",
                       length = 120)

EC_AD_BC = finder.Barcode(name = "AD_BC",
                       preceder = "CGCGCC",
                       post = "",
                       length = 11)

EC_RPTR_BC = finder.Barcode(name = "RPTR_BC",
                       preceder = "CTCGAG",
                       post = "",
                       length = 14)
AD_objects = [EC_AD, EC_AD_BC]


AD_UMI = finder.Barcode(name = "UMI",
                        preceder = "",
                       post = "",
                       length = 18)

AD_UMI_objects = [EC_AD, EC_AD_BC, AD_UMI]

In [11]:
db_path = "../../duckdb/GCN4_umi_discrepancy_umi_flank.db"

In [12]:
for file_path in yeast_pool_C_umi_AD_seq_files:
    
    # Get the file naeme to use for database
    base_name = os.path.basename(file_path)
    name_only = base_name.split('.')[0]
    print(name_only) 

    # # Get the file naeme to use for output
    # umi_path = os.path.join(yeast_pool_C_umi_output_path, f"trebl_experiment_yeast_pool_C_umi_{name_only}")
    # print(umi_path)

    # Extract UMIs and barcodes from reads
    umi_mapper = initial_map.InitialMapper(db_path = db_path,
                                       step_name = f"trebl_experiment_yeast_pool_C_umi_{name_only}", 
                                       seq_file = file_path,
                                       design_file_path = "/global/scratch/projects/fc_mvslab/OpenProjects/EChase/TREBLEseq_ismaybethenewcibername/A10_sequencing/v2/current/a10_designfile.csv",
                                       bc_objects = AD_objects,
                                        umi_object=AD_UMI,
                                       reverse_complement = True)
    umi_mapper.create_map()
    
    # Only keep barcodes of correct length
    refiner = map_refiner.MapRefiner(db_path = db_path,
                                        bc_objects=AD_objects,
                                        column_pairs = [],
                                        map_order = ['quality', 'designed'],
                                        step_name=f"trebl_experiment_yeast_pool_C_umi_{name_only}", 
                                        descriptor = "",
                                        output_figures_path='/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/TREBL/output/GCN4/yeast_pool_C_umi/figures',
                                        reads_threshold = 0,
                                        umi_object = AD_UMI)
    # refiner.refine_map_from_db()
    # refiner.plot_loss()

    # # Run both deduplications
    # deduplicator = umi_deduplicate.UMIDeduplicator(db_path = db_path,
    #                                                     bc_objects = AD_objects,
    #                                                     step_name = f"trebl_experiment_yeast_pool_C_umi_{name_only}", 
    #                                                     descriptor = "",
    #                                                     step1_map_name = None,
    #                                                     fastq_path = file_path,
    #                                                     output_path = umi_path, 
    #                                                    refined_map_suffix = 'designed')

    # deduplicator.run_simple_deduplication()
    # deduplicator.save_simple_deduplication()
    

AD_1_0
Reading 1 FASTQ/TXT file(s)...
Done in 5.32 minutes.

Reverse complement of sequences...
Done in 1.33 minutes.

Extracting 2 barcodes...
AD: extracting 120 bases after preceder 'GGCTAGC'
AD_BC: extracting 11 bases after preceder 'CGCGCC'
Done in 51.57 seconds.

Extracting UMI...
UMI: extracting last 18 bases
Done in 7.33 seconds.

Merging with design file...
Done in 32.15 seconds.

Mapping complete.
Base prefix (stable across descriptors): trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_
Full prefix for this instance: trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_

Using the following step order:
1. initial
2. quality
3. designed



In [ ]:
# sampled_rows = refiner.get_map_df('initial')
# sampled_rows["umi_flank"] = sampled_rows["UMI"].str[:6]
# sampled_rows["umi_flank"].value_counts()

# Analyzing output

In [2]:
con = duckdb.connect("../../duckdb/GCN4_umi_discrepancy_umi_flank.db")

In [3]:
con.execute("""
SELECT * 
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
LIMIT 10
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed
0,ATGCCAGCATTTGCTCAAGCTGGTGGATTTACTTTGGAACAAGTTT...,True,TACGTGTGAAG,True,TGATTTAGTCAAGCAGCC,True,1
1,TCTACTGCATTTACTAATTTGACTTCTCCATCTACTTATAATGAAT...,True,GAACCCTCTGA,True,TGATTTTAGTGTGGCGAC,True,1
2,TTGGTTCAAGAACCATTTATGTCTGCTCCAAATTCTGCTGCTTTGA...,True,GTCCCCCCGAC,True,TGATTTCGACCAATTCCA,True,1
3,ACTTTGGATGTTGCTCCAGCTGCTGTTTCTAATCCACAAGAAACTG...,True,CGCCAAGAATG,True,TGATTTTCTCCTTACAAC,True,1
4,TTGTATGATTATCCATCATTTGTTACTGATACTCCACAAACTTGGT...,True,GGCAAAGCTGG,True,TGATTTTACCCGGCCCAT,True,1
5,CATTCTGTTGAAGCTTCTCCAGCTACTCCATCTGAAGATTTGGAAG...,True,TCCTCCTTATG,True,TGATTTGCACCCGGCTGT,True,1
6,TTGGGTGAATTGGTCTTTGAGAAATTTGCTTGTGCTGATAATTTGG...,True,GGGCTCGTTGA,True,TGATTTAACCTTGCAACC,True,1
7,AGAGCTAGAAATACTTTGGCTGCTAGGAAATCTAGACAAAGGAAGA...,True,TGCTGGTGACT,True,TGATTTTGACTTAAGTTG,True,1
8,CAACAATCTCAATTGTTTACTCCAAATCCATCTTCTACTTTGCCAA...,True,AGCAGTATATC,True,TGATTTTTCTAGTGTAGC,True,1
9,TTTATGGATGCTTCTGCTCCACCATCTGCTTCATTTACTGATTTGT...,True,CCCATCCTCTC,True,TGATTTCCGCCTTAGAAA,True,1


In [4]:
con.execute("""
CREATE TABLE trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial_split AS
SELECT
    *,
    SUBSTR(UMI, 1, 6) AS flank,
    SUBSTR(UMI, LENGTH(UMI) - 11, 12) AS umi,
    UMI AS flank_and_UMI
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial;
""")

In [5]:
con.execute("""
SELECT * FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial_split LIMIT 10;
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed,flank,umi_1,flank_and_UMI
0,ATGCCAGCATTTGCTCAAGCTGGTGGATTTACTTTGGAACAAGTTT...,True,TACGTGTGAAG,True,TGATTTAGTCAAGCAGCC,True,1,TGATTT,AGTCAAGCAGCC,TGATTTAGTCAAGCAGCC
1,TCTACTGCATTTACTAATTTGACTTCTCCATCTACTTATAATGAAT...,True,GAACCCTCTGA,True,TGATTTTAGTGTGGCGAC,True,1,TGATTT,TAGTGTGGCGAC,TGATTTTAGTGTGGCGAC
2,TTGGTTCAAGAACCATTTATGTCTGCTCCAAATTCTGCTGCTTTGA...,True,GTCCCCCCGAC,True,TGATTTCGACCAATTCCA,True,1,TGATTT,CGACCAATTCCA,TGATTTCGACCAATTCCA
3,ACTTTGGATGTTGCTCCAGCTGCTGTTTCTAATCCACAAGAAACTG...,True,CGCCAAGAATG,True,TGATTTTCTCCTTACAAC,True,1,TGATTT,TCTCCTTACAAC,TGATTTTCTCCTTACAAC
4,TTGTATGATTATCCATCATTTGTTACTGATACTCCACAAACTTGGT...,True,GGCAAAGCTGG,True,TGATTTTACCCGGCCCAT,True,1,TGATTT,TACCCGGCCCAT,TGATTTTACCCGGCCCAT
5,CATTCTGTTGAAGCTTCTCCAGCTACTCCATCTGAAGATTTGGAAG...,True,TCCTCCTTATG,True,TGATTTGCACCCGGCTGT,True,1,TGATTT,GCACCCGGCTGT,TGATTTGCACCCGGCTGT
6,TTGGGTGAATTGGTCTTTGAGAAATTTGCTTGTGCTGATAATTTGG...,True,GGGCTCGTTGA,True,TGATTTAACCTTGCAACC,True,1,TGATTT,AACCTTGCAACC,TGATTTAACCTTGCAACC
7,AGAGCTAGAAATACTTTGGCTGCTAGGAAATCTAGACAAAGGAAGA...,True,TGCTGGTGACT,True,TGATTTTGACTTAAGTTG,True,1,TGATTT,TGACTTAAGTTG,TGATTTTGACTTAAGTTG
8,CAACAATCTCAATTGTTTACTCCAAATCCATCTTCTACTTTGCCAA...,True,AGCAGTATATC,True,TGATTTTTCTAGTGTAGC,True,1,TGATTT,TTCTAGTGTAGC,TGATTTTTCTAGTGTAGC
9,TTTATGGATGCTTCTGCTCCACCATCTGCTTCATTTACTGATTTGT...,True,CCCATCCTCTC,True,TGATTTCCGCCTTAGAAA,True,1,TGATTT,CCGCCTTAGAAA,TGATTTCCGCCTTAGAAA


In [8]:
con.execute("""
SELECT *
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial_split
WHERE AD_BC = 'CTGTTGTGAAA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
LIMIT 20
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed,flank,umi_1,flank_and_UMI
0,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCTATGTTGGGCC,True,1,TGATTT,CTATGTTGGGCC,TGATTTCTATGTTGGGCC
1,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAGCCCATCCGTA,True,1,TGATTT,AGCCCATCCGTA,TGATTTAGCCCATCCGTA
2,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTCCGTACCCCTC,True,1,TGATTT,TCCGTACCCCTC,TGATTTTCCGTACCCCTC
3,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTCCGTACCCCTC,True,1,TGATTT,TCCGTACCCCTC,TGATTTTCCGTACCCCTC
4,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCAACAATGCGTT,True,1,TGATTT,CAACAATGCGTT,TGATTTCAACAATGCGTT
5,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCAACAATGCGTT,True,1,TGATTT,CAACAATGCGTT,TGATTTCAACAATGCGTT
6,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCCCGACCAGAGA,True,1,TGATTT,CCCGACCAGAGA,TGATTTCCCGACCAGAGA
7,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAAGGCAGAACTC,True,1,TGATTT,AAGGCAGAACTC,TGATTTAAGGCAGAACTC
8,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAAGGCAGAACTC,True,1,TGATTT,AAGGCAGAACTC,TGATTTAAGGCAGAACTC
9,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTTAACCCCGAGG,True,1,TGATTT,TTAACCCCGAGG,TGATTTTTAACCCCGAGG


In [13]:
con.execute("""
SELECT *
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial_split
WHERE AD_BC = 'CTGTTGTGAAA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
LIMIT 20
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed,flank,umi_1,flank_and_UMI
0,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCTATGTTGGGCC,True,1,TGATTT,CTATGTTGGGCC,TGATTTCTATGTTGGGCC
1,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAGCCCATCCGTA,True,1,TGATTT,AGCCCATCCGTA,TGATTTAGCCCATCCGTA
2,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTCCGTACCCCTC,True,1,TGATTT,TCCGTACCCCTC,TGATTTTCCGTACCCCTC
3,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTCCGTACCCCTC,True,1,TGATTT,TCCGTACCCCTC,TGATTTTCCGTACCCCTC
4,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCAACAATGCGTT,True,1,TGATTT,CAACAATGCGTT,TGATTTCAACAATGCGTT
5,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCAACAATGCGTT,True,1,TGATTT,CAACAATGCGTT,TGATTTCAACAATGCGTT
6,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTCCCGACCAGAGA,True,1,TGATTT,CCCGACCAGAGA,TGATTTCCCGACCAGAGA
7,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAAGGCAGAACTC,True,1,TGATTT,AAGGCAGAACTC,TGATTTAAGGCAGAACTC
8,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTAAGGCAGAACTC,True,1,TGATTT,AAGGCAGAACTC,TGATTTAAGGCAGAACTC
9,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,TGATTTTTAACCCCGAGG,True,1,TGATTT,TTAACCCCGAGG,TGATTTTTAACCCCGAGG


In [14]:
distinct_umi_count = con.execute("""
    SELECT COUNT(DISTINCT UMI) AS n_distinct_umis
    FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial_split
    WHERE AD_BC = 'CTGTTGTGAAA'
      AND AD_qual = 1
      AND AD_BC_qual = 1
      AND Designed = 1
      AND UMI_qual = 1
""").fetchdf()

print(distinct_umi_count)

# Yay! This matches the expectation!

   n_distinct_umis
0              493


In [10]:
con = duckdb.connect("../../duckdb/GCN4_umi_discrepancy.db", read_only=True)

In [12]:
tables_df = con.execute("SHOW TABLES").fetchdf()
tables_df['name'].iloc[1]#[tables_df["name"].str.contains("trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial")].iloc[0].iloc[0]

'trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial'

In [13]:
con.execute("""
SELECT * 
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
LIMIT 10
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed
0,GGTGAATGCTTTACTTCTAGAACTCCAGCTTCTTGTGGTGTTGCTG...,True,CCCCCTCTGCT,True,TGACGCCAACCC,True,1
1,GGTGAATGCTTTACTTCTAGAACTCCAGCTTCTTGTGGTGTTGCTG...,True,CCCCCTCTGCT,True,TGACGCCAACCC,True,1
2,CCATCTGACTTTCCATTTCCAGAAACTTTGACTTCTTCTACTGCTT...,True,TTATTCGAATT,True,CCAAGAAAACAG,True,1
3,GGTCATCAAGTCTTGCAAGGTGTTTCTCATCCACCATGGAGATCTA...,True,AGAGCAGGTGA,True,AACTGATAGTGC,True,1
4,GCTGCTAGGAAATCTAGAGCTAAGAAATTGGAAAGAATGGAAGAAA...,True,GCGTGACAAAA,True,CTCCCATCCGGG,True,1
5,AGACAAATTTCTCAACAACAACAACAACAACAACAACAACAACAAT...,True,CCTGTACCCCT,True,ATAACTACAGGA,True,1
6,AGACAAATTTCTCAACAACAACAACAACAACAACAACAACAACAAT...,True,CCTGTACCCCT,True,ATAACTACAGGA,True,1
7,TCTAAGAGATCTTCTATGTGTGGTGTTAGGAAGAGATCTCAACCAT...,True,ACTACCGACAC,True,ACACTAAAATAA,True,1
8,GGTTCTTCTTCTTCTTCTATTGGTTCTTTGGCTACTATTTCTCCAC...,True,CCCATAGCCGT,True,CCAGTTTCCACA,True,1
9,TGGGAAGGTATGGAATTGGAAGGTGTTGAAGGTGTTGAAGGTAAAT...,True,GAGGGACCGAC,True,CTGAGGGACGAA,True,1


In [21]:
mismatches[(mismatches["rep"] == 1) & (mismatches["time"] == 0)]

,Unnamed: 0,AD,AD_BC,RPTR_BC,rep,time,EC_AD_umi_count,EC_RT_umi_count,EC_activity,AD_umi_count,RT_umi_count,SK_activity
41730,41730,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,CTGTTGTGAAA,GGAAGAAGTCATGA,1,0,484,125,0.258264,1,123,123.0
41731,41731,CAACAATTGGATAAAGTTCAAGCATTTGTTTCTGGTACTGTTTCTC...,TAAACGTGAAA,TTTCCTAGGCCTTT,1,0,342,76,0.222222,1,74,74.0
41733,41733,TCTACTCCATCATTTGAATCTCCAGGTTACTTCTCTCATGATACTT...,TAAAATCGGGT,GACTCAGAAGAGTC,1,0,124,45,0.362903,1,44,44.0
41734,41734,TATAATAAAGCTTTGCAATGGGGTGATGTTATTAAATTGGTTGGTG...,GTTAAAGCTGT,GTACAAGCGTATCA,1,0,499,54,0.108216,1,55,55.0
41735,41735,GCTAGAAATACTGAAGCTGCTAGAAGATCTAGAGCTAGGAAGATGC...,ATGAATTATTG,CGATATGTATTACA,1,0,868,133,0.153226,1,134,134.0
...,...,...,...,...,...,...,...,...,...,...,...,...
55278,55278,TCTGAATCTAAGAGATCTACTGATTTGGATTCTGCTGTTGAGAACT...,TAACCCTCGCA,AAACTGCAGAAATA,1,0,270,60,0.222222,1,64,64.0
55292,55292,TTGATTTCTGCTTTGCCAGCTATGGAGAAATCTATTAATAGAGCTC...,CCCCCACGCTA,CGCCACACGTCGCA,1,0,6,15,2.500000,1,16,16.0
55295,55295,TTGCCACCAATCTTGGTTGATGAGAATGATCCAGTTGCTTTGAAGA...,ACAGTCGTACT,CTTATTCTAAACCA,1,0,515,33,0.064078,1,33,33.0
55305,55305,GTTACTTTGGAAGATGTTTCTGATTGTGCTAATGCTGTTACTTTGG...,TGAGAGACCAT,CTCTGTTTTAGAAT,1,0,86,16,0.186047,1,16,16.0


In [30]:
con.execute("""
SELECT COUNT(DISTINCT umi) AS unique_umis
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
WHERE AD_BC = 'CTGTTGTGAAA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
""").fetchdf()

,unique_umis
0,489


In [31]:
con.execute("""
SELECT *
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
WHERE AD_BC = 'CTGTTGTGAAA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
LIMIT 5
""").fetchdf()

,AD,AD_qual,AD_BC,AD_BC_qual,UMI,UMI_qual,Designed
0,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,GCAAGCATCATC,True,1
1,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,GCGCCCATTGTA,True,1
2,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,CTATGTTGGGCC,True,1
3,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,CTATGTTGGGCC,True,1
4,GTTGCTGAATCATTTGAAACTTCTCCATTGTTTGCTAATGCTGATG...,True,CTGTTGTGAAA,True,CAGAGATGGAAC,True,1


In [32]:
con.execute("""
SELECT COUNT(DISTINCT umi) AS unique_umis
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
WHERE AD_BC = 'ACCCATCCAGA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
""").fetchdf()

,unique_umis
0,599


In [28]:
con.execute("""
SELECT COUNT(DISTINCT umi)
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
WHERE AD_BC = 'TCCAAACCGAA'
  AND AD_qual = 1
  AND AD_BC_qual = 1
  AND Designed = 1
  AND UMI_qual = 1
""").fetchone()

(183,)

In [20]:
con.execute("""
SELECT COUNT(DISTINCT umi)
FROM trebl_experiment_yeast_pool_C_umi_AD_1_0_AD_AD_BC_initial
WHERE AD_BC = 'TAAAATCGGGT'
""").fetchone()

(128,)